# LP3 Machine Learning Practical 5 - K-Nearest Neighbors on Diabetes Dataset

**Problem statement:** Implement K-Nearest Neighbors algorithm on diabetes.csv dataset. Compute confusion matrix, accuracy, error rate, precision and recall on the given dataset. Dataset: https://www.kaggle.com/datasets/abdallamahgoub/diabetes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Step 1: Upload diabetes.csv manually (Google Colab only; skip this line if the file is already available)
from google.colab import files
uploaded = files.upload()

# Step 2: Load dataset
df = pd.read_csv("/content/diabetes.csv")
df.head()

In [ ]:
# Display basic info
print(df.info())
print(df.describe())

# Check for missing values
print(df.isnull().sum())

# Separate features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split dataset (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
sns.countplot(x='Outcome', data=df)
plt.title("Distribution of Diabetic vs Non-Diabetic")
plt.xlabel("Outcome (0 = Non-Diabetic, 1 = Diabetic)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Choose number of neighbors
k = 5

# Initialize model
knn = KNeighborsClassifier(n_neighbors=k)

# Train model
knn.fit(X_train, y_train)

# Predict
y_pred = knn.predict(X_test)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("Error Rate:", round(error_rate, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-Score:", round(f1, 4))

In [ ]:
error = []
for i in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    error.append(np.mean(pred_i != y_test))

plt.figure(figsize=(10,5))
plt.plot(range(1, 21), error, marker='o', color='blue')
plt.title('Error Rate vs K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Error')
plt.show()

## Theory and Explanation
The last cell contains the theory, steps, result interpretation and viva questions as comments.

In [ ]:
# =====================================================================
# THEORY AND EXPLANATION - Practical 5: KNN on the Diabetes Dataset
# =====================================================================
#
# 1. PROBLEM
# Predict whether a patient is diabetic (Outcome = 1) or not (Outcome = 0) from
# medical measurements, using K-Nearest Neighbors. Compute the confusion matrix,
# accuracy, error rate, precision and recall. Binary CLASSIFICATION.
#
# 2. DATASET
# 768 patients, 8 features: Pregnancies, Glucose, BloodPressure, SkinThickness,
# Insulin, BMI, Pedigree (diabetes pedigree function), Age. Target: Outcome.
# About 500 non-diabetic and 268 diabetic, so the classes are imbalanced.
#
# 3. KEY THEORY
# a) K-Nearest Neighbors
#    A lazy, instance-based learner. There is no training phase; the training data
#    is simply stored. To classify a new patient:
#      1. compute the Euclidean distance d = sqrt( sum (xi - yi)^2 ) to every
#         training patient,
#      2. pick the k closest patients,
#      3. predict the majority class among them.
#    Small k: flexible but noisy (overfits). Large k: smoother but may underfit.
#    An odd k avoids ties in binary classification.
#
# b) Why feature scaling matters
#    KNN is distance based. Insulin ranges up to 846 while Pedigree is below 3, so
#    without scaling Insulin would dominate the distance. StandardScaler converts
#    every feature to mean 0 and standard deviation 1.
#
# c) Confusion matrix (diabetic is the positive class)
#    TP: diabetic predicted diabetic       TN: healthy predicted healthy
#    FP: healthy predicted diabetic        FN: diabetic predicted healthy
#    Accuracy   = (TP + TN) / total
#    Error rate = 1 - Accuracy = (FP + FN) / total
#    Precision  = TP / (TP + FP)
#    Recall     = TP / (TP + FN)     (also called sensitivity)
#    F1-score   = 2 * P * R / (P + R)
#
# 4. STEPS (code flow)
#    1. Load diabetes.csv, check info, statistics and missing values.
#    2. Split features X and target y, scale X with StandardScaler.
#    3. 80/20 train-test split (random_state = 42). The test set has 154 patients.
#    4. Plot the class distribution.
#    5. Create KNN with k = 5, fit on the training set, predict the test set.
#    6. Plot the confusion matrix and compute accuracy, error rate, precision,
#       recall and F1.
#    7. Repeat for k = 1 to 20 and plot the error rate against k to see the best k.
#
# 5. RESULTS AND HOW TO READ THEM (k = 5)
#    Confusion matrix: TN 79, FP 20, FN 28, TP 27
#    Accuracy 0.6883, Error rate 0.3117, Precision 0.5745, Recall 0.4909, F1 0.5294
#
#    About 69 percent of predictions are right. Of the patients predicted diabetic,
#    57 percent really are (precision). The model finds only about 49 percent of the
#    actual diabetic patients (recall), which is poor for a medical screening task
#    where a missed case (false negative) is the costly mistake.
#
#    Error versus k plot: error is highest at k = 1 (about 0.38, overfitting to
#    single neighbours) and falls as k grows, reaching its lowest around k = 18
#    (about 0.234). So k = 5 is not the best choice; k near 18 gives clearly lower
#    error on this split. Caution: choosing k by looking at the test set makes the
#    result a little optimistic. The cleaner way is cross-validation on training data.
#
#    Possible improvements: in this dataset zeros in Glucose, BloodPressure,
#    SkinThickness, Insulin and BMI actually mean missing values; replacing them
#    with the median would help. Also try a tuned k, other models, and class
#    balancing.
#
# 6. VIVA QUESTIONS
#    Q: Why is KNN called a lazy learner?
#    A: It does no learning at training time; all computation happens when
#       predicting.
#    Q: How do you choose k?
#    A: Try a range of values and pick the one with the lowest validation error
#       (the elbow of the error curve), preferably with cross-validation.
#    Q: What distance is used?
#    A: Euclidean by default (Minkowski with p = 2). Manhattan is another option.
#    Q: Why scale the data?
#    A: Otherwise features with large ranges dominate the distance.
#    Q: Difference between precision and recall?
#    A: Precision asks how many predicted positives are correct; recall asks how
#       many real positives were found.
#    Q: What is the error rate?
#    A: 1 - accuracy, the fraction of wrong predictions.